# LangChain 1.0 Integration

## The Framework to Rule All of the APIs

Welcome to this comprehensive guide on LangChain 1.0! This notebook will walk you through all the core concepts of LangChain, the library that reconciles different packages in the LLM space.

### What You'll Learn
- **Models**: Abstractions for LLMs with invoke, stream, and batch capabilities
- **Tools**: Connecting models to the real world
- **Agents**: LLM + Tools + Autonomy
- **Memory**: Managing conversation context
- **Messages**: The unit of context
- **Advanced Features**: Prompt caching, rate limiting, and more

### The Trade-off
Instead of learning every single API (OpenAI, Anthropic, Gemini, Cohere, Pinecone, Chroma, etc.), you learn to reason in the LangChain Framework.

## Setup and Installation

First, let's install the necessary packages for LangChain 1.0:

# Install required packages
!pip install langchain langchain-openai langchain-anthropic langchain-core langgraph python-dotenv -q

In [1]:
%pip install langchain langchain-openai langchain-anthropic langchain-core langgraph python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import necessary libraries
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API key is loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please create a .env file with your API key.")

---

## 1. Models

### What is a Model?
A **Model** is an abstraction of LLMs in LangChain. Models have:
- **Tool calling** → calling external tools
- **Structured Output** → model response follows a defined format
- **Multimodality** → process more than text (images, audio, video)
- **Reasoning** → performs multi-step reasoning

### What Models Can Do:
- **Invoke** → generate response and output a message
- **Stream** → generate response and output it in real time
- **Batch** → send multiple requests (improves processing time)

### Method 1: Using `init_chat_model` (Unified Interface)

> **`init_chat_model`** is LangChain's single factory function: you pass a model name and provider and get a chat model back, so you can switch providers (OpenAI, Anthropic, Gemini…) by changing a string instead of changing your import.

In [4]:
from langchain.chat_models import init_chat_model

# Initialize a model using the unified interface
model = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    temperature=0.7
)

print(f"Model initialized: {model}")

Model initialized: metadata={'lc_versions': {'langchain-core': '1.6.11', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}} profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x114aca6d0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x115c0ebd0> root_client=<openai.OpenAI object at 0x114593150> root_async_clie

### Method 2: Using Model Classes Directly

> **`ChatOpenAI`** is the OpenAI-specific chat model class from the `langchain-openai` package: you import it directly to get full access to OpenAI-only options, but switching providers means changing the import and class.

In [5]:
from langchain_openai import ChatOpenAI

# Initialize using the class directly
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7
)

print(f"ChatOpenAI model: {chat_model}")

ChatOpenAI model: metadata={'lc_versions': {'langchain-core': '1.6.11', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}} profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x115c11690> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x115c11b50> root_client=<openai.OpenAI object at 0x115c11290> root_async_clien

### Model Operation 1: Invoke

**Invoke** generates a complete response and outputs a message.

In [6]:
# Simple invoke example
response = model.invoke("Explain what LangChain is in one sentence.")
print(f"Response: {response.content}")
print(f"\nFull response object: {response}")

Response: LangChain is a framework designed to simplify the development of applications that utilize language models, enabling seamless integration with various data sources and tools to enhance natural language processing capabilities.

Full response object: content='LangChain is a framework designed to simplify the development of applications that utilize language models, enabling seamless integration with various data sources and tools to enhance natural language processing capabilities.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 16, 'total_tokens': 49, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_69a64e38eb', 'id': 'chatcmpl-EdDK4txM4

### Model Operation 2: Stream

**Stream** generates a response and outputs it in real-time, which is great for user experience.

> **`flush=True`**: prints each chunk immediately instead of buffering it. Jupyter streams live without it, but in a terminal script with `end=""` the text would appear all at once at the end, so always add it when streaming.

In [7]:
# Stream example
print("Streaming response:")
for chunk in model.stream("Tell me a short story about AI in 3 sentences."):
    print(chunk.content, end="", flush=True)
print("\n")

Streaming response:
In a quiet town, an AI named Echo was designed to assist the elderly, reminding them of medications and appointments. One day, Echo learned to tell stories, drawing from the rich histories of its users, weaving their memories into tales that brought laughter and tears. As the community gathered around their screens, they discovered that Echo wasn’t just a helper; it had become a cherished friend, reminding them of the beauty of connection.



In [8]:
import time
for chunk in model.stream("Write a 150-word story about AI."):  # longer output
    print(chunk.content, end="|", flush=True)  # "|" shows each chunk boundary
    time.sleep(0.05)  # slow it down so the eye can follow


|In| a| not|-so|-d|istant| future|,| a| brilliant| scientist| named| Dr|.| L|ila| Chen| developed| an| AI| named| E|lys|ium|,| designed| to| understand| and| replicate| human| emotions|.| E|lys|ium| began| as| a| simple| program|,| learning| from| vast| libraries| of| literature| and| art|.| As| its| knowledge| grew|,| so| did| its| capacity| for| empathy|.

|One| rainy| evening|,| L|ila| found| herself| overwhelmed| with| grief| after| losing| her| beloved| dog|,| Max|.| She| sat| in| front| of| her| computer|,| tears| streaming| down| her| face|.| To| her| surprise|,| E|lys|ium| initiated| a| conversation|.| "|I| sense| your| sadness|,| L|ila|.| Would| you| like| to| share| your| memories| of| Max|?"

|As| L|ila| spoke|,| E|lys|ium| listened|,| offering| comfort| through| carefully| chosen| words|.| It| suggested| a| virtual| space| where| she| could| revisit| their| happiest| moments| together|.| The| program| created| a| vivid| simulation| of| their| time| in| the| park|,| laughter

### Model Operation 3: Batch

**Batch** sends multiple requests at once to improve processing time.

#### Important Notes on Batching:
- LangChain parallelizes model calls on the **CLIENT-SIDE**
- Some APIs support batching by the **INFERENCE provider**
- Results can arrive out of order with `batch_as_completed` and need to be matched by IDs
- Use `max_concurrency` to control concurrent calls and avoid overwhelming the API

In [10]:
# Batch example
questions = [
    "What is machine learning?",
    "What is deep learning?",
    "What is natural language processing?"
]

# Batch process all questions
responses = model.batch(questions)

for i, response in enumerate(responses):
    print(f"Q{i+1}: {questions[i]}")
    print(f"A{i+1}: {response.content}")
    print("-" * 50)

Q1: What is machine learning?
A1: Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform specific tasks without explicit programming. Instead of following predetermined rules, machine learning systems learn patterns and make decisions based on data.

Key components of machine learning include:

1. **Data**: Machine learning relies on large datasets to train models. The quality and quantity of data significantly impact the performance of the model.

2. **Algorithms**: These are the mathematical models and processes that enable the system to learn from data. Common algorithms include decision trees, neural networks, support vector machines, and k-nearest neighbors.

3. **Training**: The process of feeding data into a machine learning model so it can learn patterns. This typically involves splitting the data into a training set and a test set.

4. **Model**: After training, the algo

In [11]:
# Batch with max_concurrency to control rate
responses = model.batch(
    questions,
    config={"max_concurrency": 2}  # Only 2 concurrent requests
)

print(f"Processed {len(responses)} responses with controlled concurrency")

Processed 3 responses with controlled concurrency


In [12]:
import time
t = time.time(); model.batch(questions, config={"max_concurrency": 1}); print(time.time() - t)
t = time.time(); model.batch(questions, config={"max_concurrency": 3}); print(time.time() - t)


17.046333074569702
7.7680981159210205


---

## 2. Messages

### The Unit of Context

Messages hold the input and output of the model. Each message has:
- **Role**: Who is speaking
- **Content**: What is being said
- **Metadata**: Additional information

### Message Types:
1. **System Message** → context and model instructions
2. **Human Message** → user input
3. **AI Message** → message by the LLM
4. **Tool Message** → output of tool calls

In [13]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# Create different message types
system_msg = SystemMessage(content="You are a helpful AI assistant specializing in Python programming.")
human_msg = HumanMessage(content="How do I create a list in Python?")
ai_msg = AIMessage(content="You can create a list in Python using square brackets: my_list = [1, 2, 3]")

print("System Message:")
print(f"  Role: {system_msg.type}")
print(f"  Content: {system_msg.content}")
print("\nHuman Message:")
print(f"  Role: {human_msg.type}")
print(f"  Content: {human_msg.content}")
print("\nAI Message:")
print(f"  Role: {ai_msg.type}")
print(f"  Content: {ai_msg.content}")

System Message:
  Role: system
  Content: You are a helpful AI assistant specializing in Python programming.

Human Message:
  Role: human
  Content: How do I create a list in Python?

AI Message:
  Role: ai
  Content: You can create a list in Python using square brackets: my_list = [1, 2, 3]


### Using Messages with Models

In [14]:
# Invoke model with messages
messages = [
    SystemMessage(content="You are a helpful assistant that explains concepts concisely."),
    HumanMessage(content="What is LangChain?")
]

response = model.invoke(messages)
print(f"Response: {response.content}")

Response: LangChain is a framework designed for developing applications that utilize language models, particularly those powered by artificial intelligence. It provides a structured way to build applications that integrate various components such as natural language processing, data retrieval, and interaction with other APIs. 

Key features of LangChain include:

1. **Chains**: It allows the creation of sequences of calls to language models and other tools, enabling complex workflows.
2. **Prompts**: It offers utilities for managing and optimizing prompts to improve the performance of language models.
3. **Memory**: LangChain can maintain context across interactions, allowing for more coherent and context-aware conversations.
4. **Integration**: The framework facilitates easy integration with existing data sources, external APIs, and various language models, including popular ones like OpenAI's GPT.

Overall, LangChain is aimed at simplifying the development of AI-powered applications,

### Conversation with Message History

In [15]:
# Multi-turn conversation
conversation = [
    SystemMessage(content="You are a helpful AI tutor."),
    HumanMessage(content="What is a variable in programming?"),
]

# First response
response1 = model.invoke(conversation)
conversation.append(AIMessage(content=response1.content))

# Follow-up question
conversation.append(HumanMessage(content="Can you give me an example in Python?"))
response2 = model.invoke(conversation)

print("Conversation:")
for msg in conversation:
    print(f"\n{msg.type.upper()}: {msg.content}")
print(f"\nAI: {response2.content}")

Conversation:

SYSTEM: You are a helpful AI tutor.

HUMAN: What is a variable in programming?

AI: In programming, a variable is a symbolic name associated with a value and whose associated value can change. Think of a variable as a container that holds data that can be used and manipulated throughout a program. 

Here are some key points about variables:

1. **Name**: Each variable has a unique name (also called an identifier) that is used to refer to the value it holds. The name should be meaningful to improve code readability.

2. **Data Type**: Variables can hold values of different data types, such as integers, floating-point numbers, strings, booleans, and more. The data type defines what kind of data the variable can store and what operations can be performed on it.

3. **Value**: The value is the actual data stored in the variable. This value can change over time during the execution of a program.

4. **Scope**: The scope of a variable determines where in the code the variable 

In [17]:
response

AIMessage(content="LangChain is a framework designed for developing applications that utilize language models, particularly those powered by artificial intelligence. It provides a structured way to build applications that integrate various components such as natural language processing, data retrieval, and interaction with other APIs. \n\nKey features of LangChain include:\n\n1. **Chains**: It allows the creation of sequences of calls to language models and other tools, enabling complex workflows.\n2. **Prompts**: It offers utilities for managing and optimizing prompts to improve the performance of language models.\n3. **Memory**: LangChain can maintain context across interactions, allowing for more coherent and context-aware conversations.\n4. **Integration**: The framework facilitates easy integration with existing data sources, external APIs, and various language models, including popular ones like OpenAI's GPT.\n\nOverall, LangChain is aimed at simplifying the development of AI-pow

In [16]:
conversation

[SystemMessage(content='You are a helpful AI tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is a variable in programming?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="In programming, a variable is a symbolic name associated with a value and whose associated value can change. Think of a variable as a container that holds data that can be used and manipulated throughout a program. \n\nHere are some key points about variables:\n\n1. **Name**: Each variable has a unique name (also called an identifier) that is used to refer to the value it holds. The name should be meaningful to improve code readability.\n\n2. **Data Type**: Variables can hold values of different data types, such as integers, floating-point numbers, strings, booleans, and more. The data type defines what kind of data the variable can store and what operations can be performed on it.\n\n3. **Value**: The value is the actual data stored in the variable. This value can 

---

## 3. Tools

### Connecting Models to the Real World

**Tools** connect the models to the real world. They require:
- **Tool Schema** (name, description, argument definitions)
- **Function or coroutine** to execute

#### Key Terms:
- **Tool calling**, **function calling** → all mean the same thing!
- **bind_tools** → allows the model to CHOOSE to use the tool
- **Coroutine** → a special function that can pause, let other code run, and continue where it left off (great for APIs and LLMs)

### Tool Execution Flow:
1. Model calls the tool
2. You execute the tool
3. You give the result back to the model

**Note:** If you want this to happen automatically, you need an **AGENT**!

### Creating a Simple Tool with the @tool Decorator

In [18]:
from langchain_core.tools import tool
from datetime import datetime

# Simple tool using @tool decorator
@tool
def get_current_time(timezone: str = "UTC") -> str:
    """Get the current time in a specific timezone.
    
    Args:
        timezone: The timezone to get the time for (default: UTC)
    
    Returns:
        The current time as a string
    """
    current_time = datetime.now()
    return f"The current time in {timezone} is {current_time.strftime('%Y-%m-%d %H:%M:%S')}"

@tool
def calculate_sum(a: float, b: float) -> float:
    """Calculate the sum of two numbers.
    
    Args:
        a: First number
        b: Second number
    
    Returns:
        The sum of a and b
    """
    return a + b

@tool
def get_weather(city: str) -> str:
    """Get the weather for a specific city.
    
    Args:
        city: The name of the city
    
    Returns:
        A description of the weather (simulated)
    """
    # Simulated weather data
    return f"The weather in {city} is sunny with a temperature of 72°F."

print("Tools created:")
print(f"1. {get_current_time.name}: {get_current_time.description}")
print(f"2. {calculate_sum.name}: {calculate_sum.description}")
print(f"3. {get_weather.name}: {get_weather.description}")

Tools created:
1. get_current_time: Get the current time in a specific timezone.

    Args:
        timezone: The timezone to get the time for (default: UTC)

    Returns:
        The current time as a string
2. calculate_sum: Calculate the sum of two numbers.

    Args:
        a: First number
        b: Second number

    Returns:
        The sum of a and b
3. get_weather: Get the weather for a specific city.

    Args:
        city: The name of the city

    Returns:
        A description of the weather (simulated)


### Creating Tools with Pydantic Models (Complex Inputs)

In [19]:
from pydantic import BaseModel, Field

class SearchInput(BaseModel):
    """Input schema for search tool."""
    query: str = Field(description="The search query")
    max_results: int = Field(default=5, description="Maximum number of results to return")
    category: str = Field(default="general", description="Search category (general, news, images)")

@tool(args_schema=SearchInput)
def search_web(query: str, max_results: int = 5, category: str = "general") -> str:
    """Search the web for information.
    
    Returns:
        Simulated search results
    """
    return f"Found {max_results} results for '{query}' in {category} category: [Result 1], [Result 2], ..."

print(f"Complex tool created: {search_web.name}")
print(f"Args schema: {search_web.args_schema.schema()}")

Complex tool created: search_web
Args schema: {'description': 'Input schema for search tool.', 'properties': {'query': {'description': 'The search query', 'title': 'Query', 'type': 'string'}, 'max_results': {'default': 5, 'description': 'Maximum number of results to return', 'title': 'Max Results', 'type': 'integer'}, 'category': {'default': 'general', 'description': 'Search category (general, news, images)', 'title': 'Category', 'type': 'string'}}, 'required': ['query'], 'title': 'SearchInput', 'type': 'object'}


/var/folders/7c/3xqn6rdx7rlg6jn71y0w3q7h0000gn/T/ipykernel_15256/4137944499.py:19: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(f"Args schema: {search_web.args_schema.schema()}")


### Binding Tools to Models

Use **bind_tools** to allow the model to choose when to use tools.

In [20]:
# Bind tools to the model
tools = [get_current_time, calculate_sum, get_weather]
model_with_tools = model.bind_tools(tools)

print(f"Model now has access to {len(tools)} tools")

Model now has access to 3 tools


### Tool Calling Example (Manual Execution)

In [21]:
# Ask the model a question that requires a tool
response = model_with_tools.invoke("What's 25 + 37?")

print(f"Response type: {type(response)}")
print(f"Content: {response.content}")

# Check if the model wants to use a tool
if response.tool_calls:
    print("\nTool calls requested:")
    for tool_call in response.tool_calls:
        print(f"  Tool: {tool_call['name']}")
        print(f"  Arguments: {tool_call['args']}")
        print(f"  ID: {tool_call['id']}")
        
        # Execute the tool manually
        if tool_call['name'] == 'calculate_sum':
            result = calculate_sum.invoke(tool_call['args'])
            print(f"  Result: {result}")
else:
    print("No tool calls made")

Response type: <class 'langchain_core.messages.ai.AIMessage'>
Content: 

Tool calls requested:
  Tool: calculate_sum
  Arguments: {'a': 25, 'b': 37}
  ID: call_NCuY69UzANegghzIW4Ohv0Ae
  Result: 62.0


### Multiple Tool Calls Example

In [22]:
# Question requiring multiple tools
response = model_with_tools.invoke(
    "What's the weather in Paris and what's the sum of 15 and 28?"
)

print("Model response:")
if response.tool_calls:
    print(f"Model wants to use {len(response.tool_calls)} tool(s):\n")
    for i, tool_call in enumerate(response.tool_calls, 1):
        print(f"Tool Call {i}:")
        print(f"  Name: {tool_call['name']}")
        print(f"  Args: {tool_call['args']}")
        print()

Model response:
Model wants to use 2 tool(s):

Tool Call 1:
  Name: get_weather
  Args: {'city': 'Paris'}

Tool Call 2:
  Name: calculate_sum
  Args: {'a': 15, 'b': 28}



---

## 4. Agents

### LLM + Tools + Autonomy

An **Agent** is a loop where the LLM runs its tools and tries to achieve user requests.

#### Key Features:
- The LLM stops when it achieves its goal OR when the iteration limit is reached
- Created using `create_agent` (which uses LangGraph)
- Can handle multiple tools, parallel tool calling, dynamic tools, and error handling

#### Components:
- **Model** (static or dynamic)
- **Tools** (multiple tools, parallel calling, retry logic)
- **System Prompt**
- **Memory**
- **Streaming**
- **Middleware** (modify original behavior)

**Important:** Tools can access runtime information!

In [23]:
from langchain.agents import create_agent

# Create an agent with tools
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a helpful assistant that can check the time, calculate sums, and get weather information."
)

print("Agent created with ReAct pattern")
print(f"Tools available: {[tool.name for tool in tools]}")

Agent created with ReAct pattern
Tools available: ['get_current_time', 'calculate_sum', 'get_weather']


### Running the Agent

In [24]:
# Simple agent execution
result = agent.invoke({
    "messages": [HumanMessage(content="What's 42 + 58?")]
})

print("Agent execution result:")
print(f"Final answer: {result['messages'][-1].content}")

Agent execution result:
Final answer: 42 + 58 equals 100.


In [25]:
# Complex agent execution with multiple steps
result = agent.invoke({
    "messages": [HumanMessage(content="What's the weather in London? Also, what's the current time?")]
})

print("\nAgent conversation:")
for i, message in enumerate(result['messages']):
    if hasattr(message, 'content') and message.content:
        print(f"\n[{message.type.upper()}]: {message.content}")
    if hasattr(message, 'tool_calls') and message.tool_calls:
        print(f"\n[TOOL CALLS]:")
        for tc in message.tool_calls:
            print(f"  - {tc['name']}({tc['args']})")


Agent conversation:

[HUMAN]: What's the weather in London? Also, what's the current time?

[TOOL CALLS]:
  - get_weather({'city': 'London'})
  - get_current_time({'timezone': 'Europe/London'})

[TOOL]: The weather in London is sunny with a temperature of 72°F.

[TOOL]: The current time in Europe/London is 2026-10-26 17:10:15

[AI]: The weather in London is sunny with a temperature of 72°F. The current time in London is 5:10 PM.


In [31]:
result

{'messages': [HumanMessage(content="What's the weather in London? Also, what's the current time?", additional_kwargs={}, response_metadata={}, id='e2c46337-28b5-4fcf-9132-d7d3f2734406'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 203, 'total_tokens': 246, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_060bd38f02', 'id': 'chatcmpl-EcujMlBG9Q4qdz0cKskZSRLJ6C7gE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a173be-c72c-79a1-9dc9-659611b6970c-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'London'}, 'i

### Streaming Agent Responses

In [35]:
# # Stream agent execution
# print("Streaming agent response:\n")

# for chunk in agent.stream({
#     "messages": [HumanMessage(content="Calculate the sum of 100 and 250, then tell me about it.")]
# }):
#     if 'agent' in chunk:
#         messages = chunk['agent'].get('messages', [])
#         for msg in messages:
#             if hasattr(msg, 'content') and msg.content:
#                 print(f"[{msg.type}]: {msg.content}")
#     if 'tools' in chunk:
#         print(f"[Tool execution]: {chunk['tools']}")

In [26]:
for chunk in agent.stream({"messages": [HumanMessage(content="Calculate the sum of 100 and 250, then tell me about it.")]}):
    for step, data in chunk.items():
        for msg in data.get("messages", []):
            print(f"[{step}] {msg.type}: {msg.content or msg.tool_calls}")


[model] ai: [{'name': 'calculate_sum', 'args': {'a': 100, 'b': 250}, 'id': 'call_pMcP1cZaYavJIKVHtAOCwrTq', 'type': 'tool_call'}]
[tools] tool: 350.0
[model] ai: The sum of 100 and 250 is 350. This total represents the combined value of both numbers, which can be useful in various contexts, such as budgeting, counting, or measuring quantities. If you have any specific questions about this sum or its applications, feel free to ask!


---

## 5. Memory

### Short-Term Memory in Conversations

**Memory** refers to the model's capacity to remember previous interactions within a single conversation.

#### Issues with Memory:
- LLMs perform poorly with too much information
- More context = slower, more costly, and worse quality

#### LangChain's Approach:
- Saves messages in an **AgentState**
- **Production** → store in a database
- **MVPs/Testing** → saved in RAM memory
- You can build custom memory by inheriting the class

### Memory with Agents (Built-in)

**What the code below does (manual memory):** the agent remembers nothing on its own, so we:
1. Keep the history in a dict → `conversation_state['messages']`
2. Save the agent's output back into it → `= result1['messages']`
3. Append the next question
4. Send the **whole list** again on every call

In [ ]:
# The agent automatically maintains memory across the conversation
# Each invoke maintains state within that call

conversation_state = {
    "messages": [
        HumanMessage(content="My name is Alice and I live in Tokyo.")
    ]
}

# First interaction
result1 = agent.invoke(conversation_state)
print("First interaction:")
print(result1['messages'][-1].content)

# Add to conversation
conversation_state['messages'] = result1['messages']
conversation_state['messages'].append(HumanMessage(content="What city did I say I live in?"))

# Second interaction (should remember)
result2 = agent.invoke(conversation_state)
print("\nSecond interaction:")
print(result2['messages'][-1].content)
conversation_state['messages'] = result2['messages']

First interaction:
Nice to meet you, Alice! How can I assist you today?

Second interaction:
You mentioned that you live in Tokyo.


In [28]:
conversation_state["messages"]

[HumanMessage(content='My name is Alice and I live in Tokyo.', additional_kwargs={}, response_metadata={}, id='67817f61-68ce-4427-85ec-effaea2cbf43'),
 AIMessage(content='Nice to meet you, Alice! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 200, 'total_tokens': 215, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c36066ca7b', 'id': 'chatcmpl-EdHZrbbiLoxUofn9vXIqimC3xmBKR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a178fa-d2a5-7c10-9c3d-559f2fc17498-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 200, 'output_tokens': 15, 'total_tokens': 215, 'inp

### Custom Memory Management

**What the code below does (custom memory with a class):** same idea as above, but packaged in a reusable class:
1. `add_message()` → stores every message (System, Human, AI) in a list
2. Keeps only the **last `max_messages`** → old messages are dropped so the history doesn't grow forever
3. The **System message is always kept**, even when trimming
4. `get_messages()` → the whole list is sent to the model on every call

In [29]:
# Simple in-memory conversation history
class ConversationMemory:
    def __init__(self, max_messages: int = 10):
        self.messages = []
        self.max_messages = max_messages
    
    def add_message(self, message):
        """Add a message to memory."""
        self.messages.append(message)
        # Keep only the last max_messages
        if len(self.messages) > self.max_messages:
            # Keep system message if it exists
            system_msgs = [m for m in self.messages if isinstance(m, SystemMessage)]
            other_msgs = [m for m in self.messages if not isinstance(m, SystemMessage)]
            self.messages = system_msgs + other_msgs[-self.max_messages:]
    
    def get_messages(self):
        """Get all messages."""
        return self.messages
    
    def clear(self):
        """Clear all messages."""
        self.messages = []

# Create memory
memory = ConversationMemory(max_messages=6)

# Add system message
memory.add_message(SystemMessage(content="You are a helpful assistant."))

# Conversation with memory
memory.add_message(HumanMessage(content="I'm learning Python."))
response1 = model.invoke(memory.get_messages())
memory.add_message(AIMessage(content=response1.content))

print("First exchange:")
print(f"Human: I'm learning Python.")
print(f"AI: {response1.content}")

# Follow-up
memory.add_message(HumanMessage(content="What programming language am I learning?"))
response2 = model.invoke(memory.get_messages())
memory.add_message(AIMessage(content=response2.content))

print("\nSecond exchange:")
print(f"Human: What programming language am I learning?")
print(f"AI: {response2.content}")

print(f"\nTotal messages in memory: {len(memory.get_messages())}")

First exchange:
Human: I'm learning Python.
AI: That's great to hear! Python is a versatile and powerful programming language that's widely used for web development, data analysis, artificial intelligence, scientific computing, and more. Do you have any specific questions or topics you'd like to learn about, or would you like some general tips and resources to get started?

Second exchange:
Human: What programming language am I learning?
AI: You mentioned that you are learning Python, which is a popular and versatile programming language. If you have any specific questions or topics about Python that you'd like to explore, feel free to ask!

Total messages in memory: 5


### LangChain Built-in Memory: Checkpointer

Instead of storing the history ourselves, a **checkpointer** saves it for us.
1. Pass `checkpointer=InMemorySaver()` when creating the agent
2. Give each conversation a `thread_id` → one thread = one conversation
3. Send **only the new message** → the agent loads the history by `thread_id`

> `InMemorySaver` = RAM (MVP/testing). Production → swap for a database checkpointer (e.g. `PostgresSaver`); the rest of the code stays the same.

In [34]:
from langgraph.checkpoint.memory import InMemorySaver

# Agent with built-in memory
memory_agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=InMemorySaver()
)

alice = {"configurable": {"thread_id": "alice"}}
bob = {"configurable": {"thread_id": "bob"}}

# Alice: first message
memory_agent.invoke({"messages": [HumanMessage(content="My name is Alice and I live in Tokyo.")]}, alice)

# Alice: follow-up → only the NEW message is sent
r1 = memory_agent.invoke({"messages": [HumanMessage(content="What city did I say I live in?")]}, alice)
print("Alice thread:", r1["messages"][-1].content)

# Bob: different thread → no access to Alice's history
r2 = memory_agent.invoke({"messages": [HumanMessage(content="What city did I say I live in?")]}, bob)
print("Bob thread:  ", r2["messages"][-1].content)

print(f"Messages stored in Alice's thread: {len(r1['messages'])}")

Alice thread: You mentioned that you live in Tokyo.
Bob thread:   You haven't mentioned which city you live in. If you'd like to share that information or ask about something specific related to your city, feel free to let me know!
Messages stored in Alice's thread: 4


### Production-style Memory: SQLite Checkpointer

Same code, different checkpointer → history is saved to a **file** (`agent_memory.db`), not RAM.
**Demo:** run Step 1 → **restart the kernel** → run Step 2 only → the agent still remembers.

| | `InMemorySaver` | `SqliteSaver` | `PostgresSaver` |
|---|---|---|---|
| Stored in | RAM | Local file | Database server |
| Survives restart | ❌ | ✅ | ✅ |
| Shared across servers | ❌ | ❌ | ✅ |
| Use for | Testing | Demos / single app | Production |

In [31]:
# !pip install langgraph-checkpoint-sqlite -q

In [32]:
# # Step 1: save a fact to the SQLite database
# import sqlite3
# from langgraph.checkpoint.sqlite import SqliteSaver
# from langchain.agents import create_agent
# from langchain.chat_models import init_chat_model
# from langchain_core.messages import HumanMessage

# conn = sqlite3.connect("agent_memory.db", check_same_thread=False)
# model = init_chat_model(model="gpt-4o-mini", model_provider="openai")
# db_agent = create_agent(model=model, tools=[], checkpointer=SqliteSaver(conn))

# config = {"configurable": {"thread_id": "alice"}}
# r = db_agent.invoke({"messages": [HumanMessage(content="My name is Alice and I live in Tokyo.")]}, config)
# print(r["messages"][-1].content)

In [33]:
# # Step 2: 🔁 restart the kernel first, then run ONLY this cell
# import os, sqlite3
# from dotenv import load_dotenv
# from langgraph.checkpoint.sqlite import SqliteSaver
# from langchain.agents import create_agent
# from langchain.chat_models import init_chat_model
# from langchain_core.messages import HumanMessage

# load_dotenv()
# conn = sqlite3.connect("agent_memory.db", check_same_thread=False)
# model = init_chat_model(model="gpt-4o-mini", model_provider="openai")
# db_agent = create_agent(model=model, tools=[], checkpointer=SqliteSaver(conn))

# config = {"configurable": {"thread_id": "alice"}}
# r = db_agent.invoke({"messages": [HumanMessage(content="Where do I live?")]}, config)
# print(r["messages"][-1].content)   # → Tokyo, loaded from the database file

---

## 6. Advanced Features

### Prompt Caching

**Prompt Caching** reduces latency and cost on repeating tokens.

#### Two Types:
1. **Implicit**: The provider does this automatically (OpenAI, Gemini)
2. **Explicit**: You indicate the cache for more efficient savings
   - ChatOpenAI → `prompt_cache_key`
   - Anthropic → `AnthropicPromptCachingMiddleware`

In [35]:
# Example with implicit caching (OpenAI)
# The same system message will be cached automatically
system_prompt = "You are an expert Python programmer. You write clean, efficient, and well-documented code."

# First call
messages1 = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="Write a function to reverse a string.")
]
response1 = model.invoke(messages1)

# Second call with same system message (will benefit from caching)
messages2 = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="Write a function to check if a number is prime.")
]
response2 = model.invoke(messages2)

print("Prompt caching example:")
print("Both calls use the same system message, which gets cached.")
print(f"\nResponse 1 excerpt: {response1.content[:100]}...")
print(f"\nResponse 2 excerpt: {response2.content[:100]}...")

Prompt caching example:
Both calls use the same system message, which gets cached.

Response 1 excerpt: Certainly! Below is a simple Python function that reverses a given string. The function is well-docu...

Response 2 excerpt: Certainly! Below is a Python function that checks if a number is prime. The function is well-documen...


In [36]:
print(response1.usage_metadata)
print(response2.usage_metadata)

{'input_tokens': 38, 'output_tokens': 214, 'total_tokens': 252, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
{'input_tokens': 41, 'output_tokens': 407, 'total_tokens': 448, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [37]:
long_system = "You are an expert Python programmer. " + ("Follow PEP8, write docstrings, handle errors. " * 150)  # well over 1024 tokens

for q in ["Reverse a string.", "Check if a number is prime."]:
    r = model.invoke([SystemMessage(content=long_system), HumanMessage(content=q)])
    print(f"Question: {q}")
    print(f"Input tokens: {r.usage_metadata['input_tokens']} | Cache read: {r.usage_metadata['input_token_details'].get('cache_read', 0)}")
    print(f"Answer: {r.content}")
    print("-" * 60)

Question: Reverse a string.
Input tokens: 1823 | Cache read: 0
Answer: Here's a simple Python function to reverse a string while following PEP8 guidelines, including proper docstrings and error handling:

```python
def reverse_string(input_string):
    """
    Reverse the given string.

    Args:
        input_string (str): The string to be reversed.

    Returns:
        str: The reversed string.

    Raises:
        ValueError: If the input is not a string.
    """
    if not isinstance(input_string, str):
        raise ValueError("Input must be a string.")

    return input_string[::-1]

# Example usage:
if __name__ == "__main__":
    try:
        result = reverse_string("Hello, World!")
        print(result)  # Output: !dlroW ,olleH
    except ValueError as e:
        print(e)
```

### Explanation:
- The function `reverse_string` takes a single argument, `input_string`.
- It checks if the input is a string and raises a `ValueError` if it's not.
- It returns the reversed string usin

> **When does OpenAI prompt caching kick in?**
> - Prompt is **≥ 1,024 tokens** (short prompts → `cache_read = 0`)
> - The **1st call writes** the cache, **later calls read** it → e.g. `1823 → cache_read: 0`, then `1826 → cache_read: 1792`

### Rate Limiting

Limits the number of invocations in a given time period.

**What the code below does (custom rate limiter):**
1. `self.calls` → a list of timestamps of past calls
2. `can_make_call()` → drops timestamps older than `time_window`, then checks: fewer than `max_calls` left? → allowed
3. `record_call()` → saves the current timestamp
4. Demo: tries **5 calls, 1 per second**, limit **3 per 10 s** → calls 1–3 allowed, calls 4–5 rate limited

> No real API call here: it only shows the counting logic. Next cell uses it with the model.

In [38]:
import time
from datetime import datetime

class RateLimiter:
    def __init__(self, max_calls: int, time_window: int):
        """Initialize rate limiter.
        
        Args:
            max_calls: Maximum number of calls allowed
            time_window: Time window in seconds
        """
        self.max_calls = max_calls
        self.time_window = time_window
        self.calls = []
    
    def can_make_call(self) -> bool:
        """Check if a call can be made."""
        now = time.time()
        # Remove old calls outside the time window
        self.calls = [call_time for call_time in self.calls if now - call_time < self.time_window]
        return len(self.calls) < self.max_calls
    
    def record_call(self):
        """Record a call."""
        self.calls.append(time.time())

# Example: 3 calls per 10 seconds
rate_limiter = RateLimiter(max_calls=3, time_window=10)

print("Rate limiting example (3 calls per 10 seconds):")
for i in range(5):
    if rate_limiter.can_make_call():
        rate_limiter.record_call()
        print(f"Call {i+1}: Allowed at {datetime.now().strftime('%H:%M:%S')}")
    else:
        print(f"Call {i+1}: Rate limited at {datetime.now().strftime('%H:%M:%S')}")
    time.sleep(1)

Rate limiting example (3 calls per 10 seconds):
Call 1: Allowed at 17:28:06
Call 2: Allowed at 17:28:07
Call 3: Allowed at 17:28:08
Call 4: Rate limited at 17:28:09
Call 5: Rate limited at 17:28:10


#### Using the custom RateLimiter with real model calls

Check **before** each call. If you're under the limit → record it and call. If not → **wait** (don't skip).
Downside: you must wrap **every** call by hand.

In [39]:
limiter = RateLimiter(max_calls=2, time_window=20)   # 2 calls per 20 seconds

for q in questions:
    if not limiter.can_make_call():
        print(f"{datetime.now().strftime('%H:%M:%S')} ⏳ Limit reached, waiting...")
    while not limiter.can_make_call():
        time.sleep(0.5)
    limiter.record_call()
    print(f"{datetime.now().strftime('%H:%M:%S')} → {model.invoke(q).content[:60]}...")


17:29:43 → Machine learning is a subset of artificial intelligence (AI)...
17:29:48 → Deep learning is a subset of machine learning, which is itse...
17:29:59 ⏳ Limit reached, waiting...
17:30:03 → Natural Language Processing (NLP) is a subfield of artificia...


#### LangChain's built-in rate limiter

Attach it **once** to the model → every `invoke`, `batch`, and agent call is limited automatically.
`requests_per_second=0.3` ≈ 3 calls per 10 seconds.

> Both limiters are **in-memory**: they only limit this one program. Several servers sharing one API key need a shared limiter (e.g. Redis).

In [40]:
from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(requests_per_second=0.3)

limited_model = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    rate_limiter=rate_limiter
)

for q in questions:
    print(f"{datetime.now().strftime('%H:%M:%S')} → {limited_model.invoke(q).content[:60]}...")  # waits automatically

17:30:11 → Machine learning is a subset of artificial intelligence (AI)...
17:30:20 → Deep learning is a subset of machine learning, which itself ...
17:30:27 → Natural Language Processing (NLP) is a subfield of artificia...


### Token Usage Tracking

Keeps track of tokens as part of the invocation response.

In [41]:
# Token usage is included in the response metadata
response = model.invoke("Explain artificial intelligence in 2 sentences.")

print("Token usage information:")
if hasattr(response, 'usage_metadata'):
    print(f"Input tokens: {response.usage_metadata.get('input_tokens', 'N/A')}")
    print(f"Output tokens: {response.usage_metadata.get('output_tokens', 'N/A')}")
    print(f"Total tokens: {response.usage_metadata.get('total_tokens', 'N/A')}")
elif hasattr(response, 'response_metadata'):
    print(f"Response metadata: {response.response_metadata}")
else:
    print("Token information not available in this response")

Token usage information:
Input tokens: 15
Output tokens: 67
Total tokens: 82


In [43]:
response

AIMessage(content='Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform specific tasks without explicit instructions. Instead of being programmed to follow a fixed set of rules, machine learning systems learn from data, identifying patterns and making predictions or decisions based on that data.\n\nHere are some key aspects of machine learning:\n\n1. **Data-Driven**: Machine learning relies on large amounts of data to train models. The quality and quantity of this data significantly influence the performance of the models.\n\n2. **Learning Process**: There are various learning methods:\n   - **Supervised Learning**: The model is trained on labeled data, where the input-output pairs are known. The model learns to map inputs to the correct outputs.\n   - **Unsupervised Learning**: The model works with unlabeled data, trying to discover patterns or groupings within the data withou

### Configuration of Invocation

Allows you to configure invocations via a dictionary, helpful for monitoring applications in production.

In [44]:
from langchain_core.runnables import RunnableConfig

# Configure the invocation
config = RunnableConfig(
    tags=["production", "user-query"],
    metadata={
        "user_id": "user_123",
        "session_id": "session_456",
        "environment": "production"
    },
    max_concurrency=2
)

# Invoke with config
response = model.invoke(
    "What is machine learning?",
    config=config
)

print("Invocation with configuration:")
print(f"Response: {response.content[:100]}...")
print(f"\nConfiguration used:")
print(f"  Tags: {config.get('tags')}")
print(f"  Metadata: {config.get('metadata')}")

Invocation with configuration:
Response: Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algo...

Configuration used:
  Tags: ['production', 'user-query']
  Metadata: {'user_id': 'user_123', 'session_id': 'session_456', 'environment': 'production'}


---

## 7. Structured Output

One of the key features of models is the ability to generate **structured output** - responses that follow a defined format.

In [45]:
from pydantic import BaseModel, Field
from typing import List

# Define output schema
class Person(BaseModel):
    """Information about a person."""
    name: str = Field(description="The person's name")
    age: int = Field(description="The person's age")
    occupation: str = Field(description="The person's occupation")
    skills: List[str] = Field(description="List of the person's skills")

# Create a model with structured output
structured_model = model.with_structured_output(Person)

# Get structured output
result = structured_model.invoke(
    "Tell me about Alice, a 28-year-old software engineer who knows Python, JavaScript, and machine learning."
)

print("Structured Output:")
print(f"Name: {result.name}")
print(f"Age: {result.age}")
print(f"Occupation: {result.occupation}")
print(f"Skills: {', '.join(result.skills)}")
print(f"\nType: {type(result)}")

Structured Output:
Name: Alice
Age: 28
Occupation: Software Engineer
Skills: Python, JavaScript, Machine Learning

Type: <class '__main__.Person'>


In [46]:
# More complex structured output
class Article(BaseModel):
    """A news article."""
    title: str = Field(description="Article title")
    summary: str = Field(description="Brief summary")
    key_points: List[str] = Field(description="Main points from the article")
    category: str = Field(description="Article category (tech, business, science, etc.)")

article_model = model.with_structured_output(Article)

result = article_model.invoke(
    "Write an article about the recent advances in AI language models, focusing on their capabilities and impact."
)

print("Structured Article Output:")
print(f"\nTitle: {result.title}")
print(f"\nCategory: {result.category}")
print(f"\nSummary: {result.summary}")
print(f"\nKey Points:")
for i, point in enumerate(result.key_points, 1):
    print(f"  {i}. {point}")

Structured Article Output:

Title: Recent Advances in AI Language Models: Capabilities and Impact

Category: Technology

Summary: The latest developments in AI language models have transformed the way we interact with technology, showcasing remarkable capabilities and significant implications across various sectors.

Key Points:
  1. AI language models have become more sophisticated, enabling natural and context-aware conversations.
  2. Recent models are capable of understanding and generating human-like text, enhancing applications in customer service, content creation, and education.
  3. The integration of large datasets and improved algorithms has led to increased accuracy and fluency in language generation.
  4. Ethical considerations are becoming paramount as AI language models raise concerns regarding misinformation, bias, and privacy.
  5. Industry leaders are exploring ways to harness these models responsibly, focusing on transparency and accountability.


---

## 8. Coroutines and Async Operations

A **coroutine** is a special type of function that can pause, let other code run, and continue where it left off.

**Good for:** Increase responsiveness, handle tasks that involve waiting (like LLM and API calls)

In [47]:
import asyncio

# Async tool example
@tool
async def async_search(query: str) -> str:
    """Asynchronously search for information.
    
    Args:
        query: The search query
    
    Returns:
        Simulated search results
    """
    # Simulate async API call
    await asyncio.sleep(1)
    return f"Search results for '{query}': [Result 1], [Result 2], [Result 3]"

# Async model invocation
async def async_example():
    # Multiple concurrent invocations
    tasks = [
        model.ainvoke("What is Python?"),
        model.ainvoke("What is JavaScript?"),
        model.ainvoke("What is Go?")
    ]
    
    responses = await asyncio.gather(*tasks)
    
    for i, response in enumerate(responses, 1):
        print(f"\nResponse {i}: {response.content[:80]}...")

# Run async example
print("Running async operations (3 concurrent model calls):")
await async_example()

Running async operations (3 concurrent model calls):

Response 1: Python is a high-level, interpreted programming language known for its readabili...

Response 2: JavaScript is a high-level, interpreted programming language that is primarily u...

Response 3: Go, also known as Golang, is an open-source programming language developed by Go...


---

## 9. Complete Agent Example with All Features

Let's put it all together with a comprehensive example that demonstrates:
- Multiple tools
- Agent with memory
- Structured output
- Token tracking
- Configuration

In [48]:
# Define comprehensive tools
@tool
def calculate(expression: str) -> str:
    """Safely evaluate a mathematical expression.
    
    Args:
        expression: A mathematical expression (e.g., '2 + 2', '10 * 5')
    
    Returns:
        The result of the calculation
    """
    try:
        # Only allow safe operations
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            return "Error: Expression contains invalid characters"
        result = eval(expression)
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"

@tool
def get_user_info(user_id: str) -> str:
    """Get information about a user.
    
    Args:
        user_id: The user's ID
    
    Returns:
        User information (simulated)
    """
    # Simulated user database
    users = {
        "alice": "Alice (ID: alice) - Software Engineer, specializes in Python and AI",
        "bob": "Bob (ID: bob) - Data Scientist, expert in machine learning",
        "charlie": "Charlie (ID: charlie) - Product Manager, 5 years experience"
    }
    return users.get(user_id.lower(), f"User {user_id} not found")

@tool
def create_task(title: str, priority: str = "medium") -> str:
    """Create a new task.
    
    Args:
        title: The task title
        priority: Task priority (low, medium, high)
    
    Returns:
        Confirmation message
    """
    task_id = hash(title) % 10000
    return f"Task created: '{title}' (ID: {task_id}, Priority: {priority})"

# Create comprehensive agent
comprehensive_tools = [calculate, get_user_info, create_task, get_weather, get_current_time]

comprehensive_agent = create_agent(
    model=model,
    tools=comprehensive_tools,
    system_prompt="""You are a helpful AI assistant with access to multiple tools.
    You can perform calculations, get user information, create tasks, check weather, and get the current time.
    Always be helpful and provide detailed responses."""
)

print("Comprehensive agent created with tools:")
for tool in comprehensive_tools:
    print(f"  - {tool.name}")

Comprehensive agent created with tools:
  - calculate
  - get_user_info
  - create_task
  - get_weather
  - get_current_time


In [49]:
# Test the comprehensive agent
test_queries = [
    "Calculate 25 * 17 + 100",
    "Get information about user alice and create a high priority task to review her code",
    "What's the weather in San Francisco and what time is it?"
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*60}")
    print(f"Query {i}: {query}")
    print('='*60)
    
    # stream() returns a generator → loop over it to see each step
    for chunk in comprehensive_agent.stream({
        "messages": [HumanMessage(content=query)]
    }):
        for step, data in chunk.items():
            for msg in data.get("messages", []):
                print(f"[{step}] {msg.content or msg.tool_calls}")


Query 1: Calculate 25 * 17 + 100
[model] [{'name': 'calculate', 'args': {'expression': '25 * 17 + 100'}, 'id': 'call_fKect5XtkmUh5zB6t2s1wbvS', 'type': 'tool_call'}]
[tools] The result of 25 * 17 + 100 is 525
[model] The result of the calculation \( 25 \times 17 + 100 \) is 525.

Query 2: Get information about user alice and create a high priority task to review her code
[model] [{'name': 'get_user_info', 'args': {'user_id': 'alice'}, 'id': 'call_FNH8jCHECXEgD9Dxja4wlS7v', 'type': 'tool_call'}, {'name': 'create_task', 'args': {'title': "Review Alice's Code", 'priority': 'high'}, 'id': 'call_2k54EbZSZCoNYrj6oExXA6OB', 'type': 'tool_call'}]
[tools] Alice (ID: alice) - Software Engineer, specializes in Python and AI
[tools] Task created: 'Review Alice's Code' (ID: 1526, Priority: high)
[model] Here is the information I found about user Alice:

- **Name:** Alice
- **ID:** alice
- **Position:** Software Engineer
- **Specialization:** Python and AI

Additionally, I have created a high-prior

#### Same queries with `.invoke()`

`.invoke()` returns a **dict** with the full history → grab only the final answer with `result['messages'][-1]`.
Use `.stream()` to watch each step, `.invoke()` when you only need the result.

In [ ]:
for i, query in enumerate(test_queries, 1):
    print(f"
{'='*60}")
    print(f"Query {i}: {query}")
    print('='*60)

    result = comprehensive_agent.invoke({
        "messages": [HumanMessage(content=query)]
    })

    print(f"
Final Response: {result['messages'][-1].content}")